# Stage 2 Notebook 55 - Exp2ZZ Anchor + topk-fixed K=3 + VFL

**Density-tuned topk_fixed.** NB52 (Exp2WW, K=8) crashed geometry to matched_iou=0.256 vs NB48's 0.544. With 8 priors per GT all chasing the same curve, gradient spread thin -- no prior got a tight fit. The matcher fix was on the right axis (pos-neg gap doubled from 0.001 -> 0.003) just with too many positives.

Exp2ZZ: K=3 instead. 3 priors per GT * 5 GT = ~12-15 positives/image, close to dynamic-k's empirical 8-12. Stable labels (Exp2WW's win), less geometric dilution.

Single config diff vs NB52 (Exp2WW exp47): `topk_fixed_per_gt: 8 -> 3`. All else identical.

### Run mode
1. `DEBUG_MODE=True` smoke.
2. `DEBUG_MODE=False` 20 epochs limit=3000.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp50_rmt_gca_anchor_topk3_vfl_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp50_rmt_gca_anchor_topk3_vfl_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp50_rmt_gca_anchor_topk3_vfl_joint_smoke.log
OK exp50_rmt_gca_anchor_topk3_vfl_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.1357 det_loss=3.2637 grad_cos=-0.2777 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.498935341835022, 'gate/lane_mean': 0.5044562220573425, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp50_rmt_gca_anchor_topk3_vfl_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: 3000
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp50_rmt_gca_anchor_topk3_vfl_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp50_rmt_gca_anchor_topk3_vfl_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp50_rmt_gca_anchor_topk3_vfl_joint_short20.tar --epochs 20 --batch-size 8 --limit-val 1000 --force-extract --print-every 50 --limit-train 3000
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp50_rmt_gca_anchor_topk3_vfl_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp50_rmt_gca_anchor_topk3_vfl_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp50_rmt_gca_anchor_topk3_vfl_joint.yaml --curve-tar /content/drive/MyDrive/EcoC

0

## What to watch in Exp2ZZ

Reference NB52 (K=8): matched_iou=0.256, gap=0.003, decoded_f1=0.043.
Reference NB48 (dynamic_k full data): matched_iou=0.544, decoded_f1=0.050.
Reference NB54 (K=8 + full data): matched_iou=0.289, decoded_f1=0.062.

Pass criteria at epoch 20:
- `val/matched_line_iou >= 0.40` (recover most of NB48's geometry).
- `pos_score - neg_score >= 0.02` (stable labels still help cls).
- `val/lane/decoded_f1 >= 0.07` (beat NB54's record of 0.062).
- `val/lane_best_f1 >= 0.20`.